# Fine-tune SmolVLM on SROIE receipts (Kaggle)

This notebook runs the GPU steps of the receipt extractor project. It checks the untouched model, does a tiny training run to confirm the pipeline works, then does the full fine-tune, evaluation, merge and quantization.

Before running it:

1. Zip the project folder without its data folder, upload the zip as a private dataset (Add Input, Upload, New Dataset) and attach it.
2. Attach the SROIE dataset (Add Input, search for "SROIE datasetv2" by urbikn). It needs the folders train and test, each with img and entities inside.
3. In the settings panel on the right, set the accelerator to GPU T4 x2 and turn Internet on. Internet needs a phone-verified Kaggle account. The model is downloaded from Hugging Face.
4. Run the cells in order.

The first run is for checks only (DO_CHECKS = True and DO_FULL_RUN = False below). Read the output of each check. When it looks right, set DO_FULL_RUN = True and use Save Version, then Save & Run All (Commit), so the long part keeps running with the browser closed. Files are only kept after a saved version finishes, so download results from the Output tab of that version.

Each step is skipped when its output already exists, so a rerun picks up where the last one stopped. Set FORCE = True to redo steps. Time estimates in this notebook are guesses, not measurements. This code was tested on a tiny random model of the same class as SmolVLM, not on the real model, so the first real run may need adjustments.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path


def setting(name, default):
    """A value you can change below. An environment variable called RX_<NAME> overrides it,
    which is how this notebook was tested offline."""
    raw = os.environ.get("RX_" + name)
    if raw is None:
        return default
    if isinstance(default, bool):
        return raw.strip().lower() in ("1", "true", "yes", "on")
    if isinstance(default, int):
        return int(raw)
    if isinstance(default, list):
        return [part.strip() for part in raw.split(",") if part.strip()]
    return raw


# What to run. First run: checks only. Second run: switch DO_FULL_RUN on.
DO_CHECKS = setting("DO_CHECKS", True)
DO_FULL_RUN = setting("DO_FULL_RUN", False)
FORCE = setting("FORCE", False)  # True redoes steps whose output already exists

# Model and image size. SmolVLM cuts each image into 512 pixel tiles and spends 64 tokens on each,
# so the longest side decides memory and time. 0 keeps the model's own default (the log prints it).
# Use the same value for every run you want to compare.
MODEL_ID = setting("MODEL_ID", "HuggingFaceTB/SmolVLM-500M-Instruct")
IMAGE_LONGEST_EDGE = setting("IMAGE_LONGEST_EDGE", 1536)

# Training. float32 is the safe choice on a T4, which has no fast bfloat16 and where the
# training loop has no float16 loss scaling. FREEZE_VISION trains only the language model
# and uses much less memory. If the GPU runs out of memory, also try GRADIENT_CHECKPOINTING
# or a smaller IMAGE_LONGEST_EDGE such as 1024.
TRAIN_DTYPE = setting("TRAIN_DTYPE", "float32")
EPOCHS = setting("EPOCHS", 3)
LEARNING_RATE = setting("LEARNING_RATE", "2e-4")
GRAD_ACCUM = setting("GRAD_ACCUM", 8)
LORA_R = setting("LORA_R", 16)
FREEZE_VISION = setting("FREEZE_VISION", True)
GRADIENT_CHECKPOINTING = setting("GRADIENT_CHECKPOINTING", False)
TINY_EXAMPLES = setting("TINY_EXAMPLES", 40)

# Evaluation. float16 is the usual GPU inference precision, so it is the fair baseline for the
# size and speed comparison with 8 bit and 4 bit. A check below tells you if it misbehaves.
EVAL_DTYPE = setting("EVAL_DTYPE", "float16")
QUANT_MODES = setting("QUANT_MODES", ["8bit", "4bit"])
LATENCY_IMAGES = setting("LATENCY_IMAGES", 50)

# The official SROIE split has 626 training and 347 test receipts. These are what your
# laptop conversion gave, and the OCR baseline was scored on the same 347.
EXPECTED_TRAIN = setting("EXPECTED_TRAIN", 626)
EXPECTED_TEST = setting("EXPECTED_TEST", 347)

# The machine.
REQUIRE_GPU = setting("REQUIRE_GPU", True)
INSTALL = setting("INSTALL", True)
CLEAN_DATA = setting("CLEAN_DATA", True)  # delete the converted image copies at the very end
INPUT = Path(setting("INPUT_DIR", "/kaggle/input"))
WORK = Path(setting("WORK_DIR", "/kaggle/working"))

PY = sys.executable
IMAGE_FLAGS = ["--image-longest-edge", IMAGE_LONGEST_EDGE] if IMAGE_LONGEST_EDGE else []
TRAIN_FLAGS = (["--freeze-vision"] if FREEZE_VISION else []) + (
    ["--gradient-checkpointing"] if GRADIENT_CHECKPOINTING else []
)
print("settings loaded")

## Find the code and set up helpers

The project is copied to the working folder, because the input folder is read only. Results are written next to it, under outputs and reports.

In [ ]:
def find_project():
    """The folder that holds scripts/finetune.py, either unpacked in the input or inside a zip there."""
    for hit in sorted(INPUT.glob("**/scripts/finetune.py")):
        return hit.parent.parent
    for archive in sorted(INPUT.glob("**/*.zip")):
        with zipfile.ZipFile(archive) as zf:
            if any(name.endswith("scripts/finetune.py") for name in zf.namelist()):
                target = WORK / "unzipped"
                zf.extractall(target)
                for hit in sorted(target.glob("**/scripts/finetune.py")):
                    return hit.parent.parent
    return None


source = find_project()
if source is None:
    raise RuntimeError(
        "Could not find the project code. Attach the dataset that holds the project zip "
        "(Add Input on the right), then run this cell again."
    )
WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / "receipt-extractor"
shutil.copytree(
    source,
    PROJECT,
    dirs_exist_ok=True,
    ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", ".git", "data", "outputs", "reports"),
)
LOGS = WORK / "logs"
LOGS.mkdir(exist_ok=True)
(PROJECT / "reports").mkdir(exist_ok=True)
print("project code:", source, "->", PROJECT)


def run(args, log_name=None):
    """Run a command from the project folder, show its output as it appears, keep a copy in logs,
    and raise an error if it fails. Returns the output text."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", CUDA_VISIBLE_DEVICES="0",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = (LOGS / f"{log_name}.log").open("w") if log_name else None
    started = time.time()
    proc = subprocess.Popen(
        args, cwd=PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        if log:
            log.write(line)
        lines.append(line)
    code = proc.wait()
    if log:
        log.close()
    print(f"[{time.time() - started:.0f} s, exit code {code}]", flush=True)
    if code != 0:
        raise RuntimeError(
            f"The command failed with exit code {code}. The end of its output:\n" + "".join(lines[-15:])
        )
    return "".join(lines)


def done(path):
    """True when a step's output already exists, so the step can be skipped."""
    if (PROJECT / path).exists() and not FORCE:
        print(f"{path} already exists, skipping this step. Delete it or set FORCE = True to redo it "
              "(needed after changing a setting).")
        return True
    return False


def evaluate(dataset, name, *extra, model=MODEL_ID, dtype=None):
    """Score a model on a data file with scripts/evaluate.py and save reports/<name>.json."""
    run(
        [PY, "scripts/evaluate.py", dataset, "--predictor", "vlm", "--model-id", model,
         "--dtype", dtype or EVAL_DTYPE, "--day-first", *IMAGE_FLAGS, *extra, "--out", f"reports/{name}.json"],
        name,
    )


def load_report(name):
    return json.loads((PROJECT / "reports" / name).read_text())


FAILED = []

## Check the GPU and the internet

In [ ]:
import torch

print("torch", torch.__version__, "built for CUDA", torch.version.cuda)
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU: {GPU_NAME}, compute capability {capability[0]}.{capability[1]}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB memory")
    try:
        x = torch.randn(256, 256, device="cuda")
        float((x @ x).sum())
        print("a test calculation on the GPU worked")
    except Exception as error:
        raise RuntimeError(
            "The GPU is visible but this PyTorch cannot run on it. This happens on the P100 with newer "
            "PyTorch builds. In the settings panel choose GPU T4 x2 as the accelerator and run again."
        ) from error
else:
    GPU_NAME = None
    message = "No GPU found. In the settings panel on the right set the accelerator to GPU T4 x2, then run again."
    if REQUIRE_GPU:
        raise RuntimeError(message)
    print("WARNING:", message)


def internet_on():
    import urllib.request

    try:
        urllib.request.urlopen("https://huggingface.co", timeout=10)
        return True
    except Exception:
        return False


if (INSTALL or not Path(MODEL_ID).exists()) and not internet_on():
    raise RuntimeError(
        "Cannot reach huggingface.co, which is needed to install packages and download the model. "
        "In the settings panel on the right turn Internet on (it needs a phone-verified Kaggle account), "
        "then run again."
    )
print("internet is on")

## Install the libraries

This installs the pinned versions the project was tested with (requirements-vlm.txt) and bitsandbytes for 8 bit and 4 bit loading. pip may print warnings about other packages on Kaggle. They can be ignored unless a later cell fails.

In [ ]:
if sys.version_info < (3, 11):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too old. The pinned transformers 5.21.0 needs Python 3.11 or newer."
    )
if INSTALL:
    run([PY, "-m", "pip", "install", "-q", "-r", "requirements-vlm.txt", "bitsandbytes"], "install")
    # Kaggle ships an old torchao (0.10.0) that newer peft refuses to work with. Nothing here uses it.
    run([PY, "-m", "pip", "uninstall", "-y", "torchao"], "uninstall_torchao")

VERSIONS = json.loads(
    run(
        [PY, "-c",
         "import json, torch, transformers, peft, accelerate\n"
         "try:\n"
         "    import bitsandbytes; bnb = bitsandbytes.__version__\n"
         "except Exception as error:\n"
         "    bnb = 'not available: ' + str(error)[:80]\n"
         "print(json.dumps(dict(torch=torch.__version__, transformers=transformers.__version__, "
         "peft=peft.__version__, accelerate=accelerate.__version__, bitsandbytes=bnb)))"],
        "versions",
    ).strip().splitlines()[-1]
)
print(VERSIONS)
if tuple(int(part) for part in VERSIONS["torch"].split("+")[0].split(".")[:2]) < (2, 6):
    raise RuntimeError(
        f"transformers 5.21.0 needs PyTorch 2.6 or newer, but this notebook has {VERSIONS['torch']}. "
        "Run pip install -U torch torchvision in a cell, restart the session, and run again."
    )
if QUANT_MODES and VERSIONS["bitsandbytes"].startswith("not available"):
    print("WARNING: bitsandbytes did not load, so the 8 bit and 4 bit steps will fail. "
          "Set QUANT_MODES = [] to skip them.")

## Prepare the SROIE data

This uses the project's own converter on the official train and test folders, the same way as on your laptop. The counts printed at the end should match the 626 and 347 receipts from there. If they do not, the test set is not the one the OCR baseline was scored on and the comparison would not be fair.

In [ ]:
def find_sroie(split):
    """(image folder, label folder) of one SROIE split under the input folder."""
    for labels in sorted(INPUT.glob(f"**/{split}/entities")):
        images = labels.parent / "img"
        if images.is_dir():
            return images, labels
    return None


def show_tree(root, depth=4):
    for path in sorted(root.glob("*")):
        print("  " * (len(path.relative_to(INPUT).parts) - 1) + path.name)
        if path.is_dir() and len(path.relative_to(INPUT).parts) < depth:
            show_tree(path, depth)


DATA = PROJECT / "data"
for split in ("train", "test"):
    out = DATA / f"sroie_{split}"
    if (out / "data.jsonl").is_file() and not FORCE:
        print(f"{split}: already converted")
        continue
    found = find_sroie(split)
    if found is None:
        print("Folders under the input directory:")
        show_tree(INPUT)
        raise RuntimeError(
            f"Could not find a {split}/entities folder with a {split}/img folder next to it. "
            "Attach the SROIE dataset (Add Input), or adjust find_sroie above to match the folders printed here."
        )
    images, labels = found
    print(f"{split}: images {images}, labels {labels}")
    run([PY, "scripts/prepare_sroie.py", "--images", images, "--labels", labels, "--out", out], f"prepare_{split}")

TRAIN_FILE = str((DATA / "sroie_train" / "data.jsonl").relative_to(PROJECT))
TEST_FILE = str((DATA / "sroie_test" / "data.jsonl").relative_to(PROJECT))
n_train = sum(1 for line in (PROJECT / TRAIN_FILE).read_text().splitlines() if line.strip())
n_test = sum(1 for line in (PROJECT / TEST_FILE).read_text().splitlines() if line.strip())
print(f"train receipts {n_train} (expected {EXPECTED_TRAIN}), test receipts {n_test} (expected {EXPECTED_TEST})")
if (n_train, n_test) != (EXPECTED_TRAIN, EXPECTED_TEST):
    print("WARNING: the counts differ from your laptop conversion. Find out why before comparing results with the OCR baseline.")

SETTINGS_RECORD = {
    "model_id": MODEL_ID, "image_longest_edge": IMAGE_LONGEST_EDGE, "train_dtype": TRAIN_DTYPE,
    "eval_dtype": EVAL_DTYPE, "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "grad_accum": GRAD_ACCUM,
    "lora_r": LORA_R, "freeze_vision": FREEZE_VISION, "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "gpu": GPU_NAME, "versions": VERSIONS, "train_receipts": n_train, "test_receipts": n_test,
}
(PROJECT / "reports" / "run_settings.json").write_text(json.dumps(SETTINGS_RECORD, indent=2))

## Part A: checks (set DO_CHECKS = False to skip)

Check 1. The untouched model on five test receipts, with the raw text it writes. Look at what it actually wrote. It was never trained for this format, so it may add words around the JSON or get fields wrong. That is expected. What matters is that the output is readable text and not empty or garbage.

In [ ]:
if DO_CHECKS:
    evaluate(TEST_FILE, "check_zero_shot", "--limit", 5, "--show-raw", 5)

Check 2. float16 against float32 on the same five receipts. If float16 gives unreadable output where float32 does not, set EVAL_DTYPE = "float32" at the top and run again.

In [ ]:
if DO_CHECKS and EVAL_DTYPE != "float32":
    evaluate(TEST_FILE, "check_zero_shot_float32", "--limit", 5, dtype="float32")
    low = load_report("check_zero_shot.json")["predictions"]
    full = load_report("check_zero_shot_float32.json")["predictions"]
    both = [(a, b) for a, b in zip(low, full) if a is not None and b is not None]
    print(f"{EVAL_DTYPE} and float32 gave identical fields on {sum(a == b for a, b in both)} of {len(both)} "
          "receipts where both outputs were readable")
    print(f"unreadable outputs: {EVAL_DTYPE} {sum(p is None for p in low)}, float32 {sum(p is None for p in full)}")
    if sum(p is None for p in low) > sum(p is None for p in full):
        print(f'{EVAL_DTYPE} failed where float32 did not. Set EVAL_DTYPE = "float32" at the top.')

Check 3. A tiny training run. The log shows how many tokens one receipt becomes (this decides memory), the image settings in use, how many parameters train, and the loss at each step. Peak GPU memory is printed at the end. The loss should go down.

In [ ]:
if DO_CHECKS and not done("outputs/tiny/adapter_config.json"):
    run(
        [PY, "scripts/finetune.py", TRAIN_FILE, "--model-id", MODEL_ID, "--output", "outputs/tiny",
         "--max-examples", TINY_EXAMPLES, "--epochs", 2, "--grad-accum", 4, "--dtype", TRAIN_DTYPE,
         *IMAGE_FLAGS, *TRAIN_FLAGS],
        "tiny_finetune",
    )
if DO_CHECKS:
    history = json.loads((PROJECT / "outputs/tiny/history.json").read_text())
    losses = [h["train_loss"] for h in history if "train_loss" in h]
    print(f"training loss: first step {losses[0]:.3f}, last step {losses[-1]:.3f}")
    print("OK, the loss went down." if losses[-1] < losses[0] else
          "The loss did not go down. Do not start the full run yet. Check the log above.")

Check 4. The tiny model on ten test receipts. After even this little training, the output should be valid JSON with the three fields. Scores can still be low. Parse failures near zero is the thing to look for.

In [ ]:
if DO_CHECKS:
    evaluate(TEST_FILE, "check_tiny", "--adapter", "outputs/tiny", "--limit", 10, "--show-raw", 3)
    report = load_report("check_tiny.json")
    print("parse failure rate:", report["metrics"]["parse_failure_rate"])

If the checks look right, set DO_FULL_RUN = True in the settings cell (and DO_CHECKS = False if you do not want to repeat them) and use Save Version, then Save & Run All (Commit). If a check looks wrong, do not start the full run. The logs folder keeps the output of every step.

## Part B: the full run (set DO_FULL_RUN = True)

Step 1. The untouched model on all test receipts. This is the "no fine-tuning" row of the results table.

In [ ]:
if DO_FULL_RUN and not done("reports/zero_shot.json"):
    evaluate(TEST_FILE, "zero_shot", "--latency", "--latency-images", LATENCY_IMAGES)

Step 2. The full fine-tune. The validation loss after each epoch is the number to watch. If it starts rising while the training loss keeps falling, the model is memorizing, and fewer epochs would be better.

In [ ]:
if DO_FULL_RUN and not done("outputs/lora/adapter_config.json"):
    run(
        [PY, "scripts/finetune.py", TRAIN_FILE, "--model-id", MODEL_ID, "--output", "outputs/lora",
         "--epochs", EPOCHS, "--lr", LEARNING_RATE, "--grad-accum", GRAD_ACCUM,
         "--lora-r", LORA_R, "--lora-alpha", 2 * LORA_R, "--dtype", TRAIN_DTYPE, *IMAGE_FLAGS, *TRAIN_FLAGS],
        "finetune",
    )
if DO_FULL_RUN:
    history = json.loads((PROJECT / "outputs/lora/history.json").read_text())
    for entry in history:
        if "val_loss" in entry:
            print(f"epoch {entry['epoch']}: validation loss {entry['val_loss']:.4f}")

Step 3. Merge the adapter into the base model, saved in float32. Then check that the merged model writes the same answers as the base model with the adapter on twenty receipts. Small differences can appear from rounding, but most receipts should match.

In [ ]:
if DO_FULL_RUN and not done("outputs/merged/config.json"):
    run([PY, "scripts/merge_adapter.py", "--model-id", MODEL_ID, "--adapter", "outputs/lora",
         "--output", "outputs/merged", "--dtype", "float32", *IMAGE_FLAGS], "merge")

if DO_FULL_RUN:
    evaluate(TEST_FILE, "merge_check_adapter", "--adapter", "outputs/lora", "--limit", 20)
    evaluate(TEST_FILE, "merge_check_merged", "--limit", 20, model="outputs/merged")
    a = load_report("merge_check_adapter.json")["predictions"]
    b = load_report("merge_check_merged.json")["predictions"]
    print(f"adapter and merged model gave identical fields on {sum(x == y for x, y in zip(a, b))} of {len(a)} receipts")

Step 4. The fine-tuned model (merged, float16 by default) on all test receipts. This is the "fine-tuned" row.

In [ ]:
if DO_FULL_RUN and not done("reports/tuned.json"):
    evaluate(TEST_FILE, "tuned", "--latency", "--latency-images", LATENCY_IMAGES, model="outputs/merged")

Step 5. The same merged model loaded in 8 bit and 4 bit (bitsandbytes), on the same receipts. Compare size, latency and accuracy against step 4. A failure here is recorded and the notebook carries on, so the rest of the results are not lost.

In [ ]:
REPORT_NAMES = {"8bit": "int8", "4bit": "int4"}
if DO_FULL_RUN:
    for mode in QUANT_MODES:
        name = REPORT_NAMES.get(mode, mode)
        if done(f"reports/{name}.json"):
            continue
        try:
            evaluate(TEST_FILE, name, "--quantization", mode, "--latency", "--latency-images", LATENCY_IMAGES,
                     model="outputs/merged")
        except RuntimeError as error:
            FAILED.append(mode)
            print(f"{mode} failed: {error}")

Step 6. The results table for the VLM rows. The OCR row comes from your laptop run (reports/ocr.json there). Latency was measured on this GPU, and the OCR latency on your laptop's CPU, so say which hardware each row used when you report it.

In [ ]:
if DO_FULL_RUN:
    names = [("zero_shot", "zero_shot.json"), ("tuned", "tuned.json"),
             ("int8", "int8.json"), ("int4", "int4.json")]
    pairs = [f"{label}=reports/{file}" for label, file in names if (PROJECT / "reports" / file).is_file()]
    table = run([PY, "scripts/compare.py", *pairs], "compare")
    (PROJECT / "reports" / "results.txt").write_text(table)
    print("image settings used:", load_report("tuned.json").get("image_settings"))
    print("evaluation precision:", EVAL_DTYPE, "| GPU:", GPU_NAME)
    if FAILED:
        print("steps that failed:", FAILED)

## Collect the results

This zips the reports, the adapter and the logs into one file in the output folder. The merged model stays in outputs/merged and is large, so download it only if you need it. For serving, the base model plus the adapter is enough.

In [ ]:
if DO_FULL_RUN:
    archive = WORK / "receipt_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder in (PROJECT / "reports", PROJECT / "outputs" / "lora", LOGS):
            for path in sorted(folder.rglob("*")):
                if path.is_file():
                    zf.write(path, path.relative_to(WORK))
    print(f"wrote {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
    if CLEAN_DATA and (PROJECT / "data").exists():
        shutil.rmtree(PROJECT / "data")
        print("removed the converted image copies (they can be rebuilt from the SROIE input)")
    print("Next: Save Version, then download receipt_results.zip from the Output tab of the finished version.")